# 03 Run Experiments

Use this notebook when you want to run models interactively instead of using the command line scripts. The default settings are intentionally smaller than the full matrix, but you can flip `RUN_FULL_MATRIX` to `True`.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

In [ ]:
import subprocess
from IPython.display import display

from scripts.make_summary_tables import summarize_results
from src.utils.config import load_experiment_config
from src.utils.io import load_dataframe, save_dataframe, save_json_atomic

## Choose Matrix Size

In [ ]:
config = load_experiment_config('configs/experiment_config.yaml')

RUN_FULL_MATRIX = False
OUTPUT_PATH = Path('results/tables/notebook_experiment_results.csv')

if RUN_FULL_MATRIX:
    datasets = config['datasets']
    noise_types = config['noise_types']
    noise_rates = config['noise_rates']
    models = config['models']
    detectors = config['detectors']
    seeds = config['seeds']
else:
    datasets = ['breast_cancer', 'wine']
    noise_types = ['symmetric', 'class_dependent']
    noise_rates = [0.1, 0.3]
    models = ['logistic_regression', 'random_forest']
    detectors = ['combined', 'confidence']
    seeds = [42, 123]

cv_folds = int(config.get('cv_folds', 5))

n_combos = len(datasets) * len(noise_types) * len(noise_rates) * len(models) * len(seeds)
rows_per_combo = 2 + 2 * len(detectors)
print(f'Combinations: {n_combos}')
print(f'Expected rows: {n_combos * rows_per_combo}')
print(f'Output: {OUTPUT_PATH}')

## Run and Save

In [ ]:
notebook_config = dict(config)
notebook_config.update(
    datasets=datasets, noise_types=noise_types, noise_rates=noise_rates,
    models=models, detectors=detectors, seeds=seeds, cv_folds=cv_folds,
)
summary_path = OUTPUT_PATH.with_name(OUTPUT_PATH.stem + '_summary.csv')
notebook_config['output'] = {
    'results_csv': str(OUTPUT_PATH), 'summary_csv': str(summary_path),
}
config_path = save_json_atomic(notebook_config, 'results/logs/notebook_config.json')
subprocess.run(
    [sys.executable, 'scripts/run_cleaning_experiments.py',
     '--config', str(config_path), '--output', str(OUTPUT_PATH)],
    cwd=PROJECT_ROOT, check=True,
)
results = load_dataframe(OUTPUT_PATH)
display(results.head())

## Notebook Summary Table

In [ ]:
summary = summarize_results(results)
save_dataframe(summary, summary_path)
print(f'Saved {summary_path}')
display(summary.sort_values('accuracy_mean', ascending=False).head(20))